# Practical Problems 3 — Graphs & Genome Assembly
## TL;DR — Plain English Introduction

**What is a graph?** Nodes (circles) connected by edges (lines). DNA sequences, protein interaction networks, evolutionary trees — all graphs.

**Why does genome assembly use graphs?** DNA sequencers give you millions of short reads (100-300bp). Genome assembly = find the original genome by overlapping reads. This is the "shortest superstring" problem — solved with De Bruijn graphs and Eulerian paths.

**No Python background?**
```python
# Graph as adjacency list (most common representation)
graph = {
    'A': ['B', 'C'],   # A connects to B and C
    'B': ['A', 'D'],   # B connects to A and D
    'C': ['A'],        # C connects to A only
    'D': ['B']         # D connects to B only
}
# BFS: explore level by level (queue)
# DFS: explore deep first (stack or recursion)
```

**What you'll solve:**
- HackerRank: BFS shortest path, DFS connected components, Dijkstra, MST
- Rosalind: Overlap graphs (GRPH), greedy assembly (LONG), De Bruijn + Eulerian (DBRU)

**Learning path:** 00/01 (Python dicts/lists) → 08/02 (DP) → This notebook → 01/03 (Assembly) → 06/01 (GNNs)

## Beginner Teaching Frame

**Who should fully work through this notebook:** every student. This is where fluency is built.

**How to study it on a first pass:**
- try each problem before reading the answer
- keep a timer for at least one section
- write the complexity and the biological interpretation after each solution

**Common traps:** reading solution code too early, confusing recognition with mastery, and skipping timed practice because the answers are available.


## Canonical Resource Upgrade

Use these as practice companions:
- [HackerRank](https://www.hackerrank.com/domains)
- [Rosalind](https://rosalind.info/problems/list-view/)
- [Big-O Cheat Sheet](https://www.bigocheatsheet.com/)


# Practical 3 — Graphs, Trees & Genome Assembly

## What This Notebook Is

Graph algorithms appear in two forms in bioinformatics:
- **Abstract graphs**: HackerRank tests BFS, DFS, shortest paths, spanning trees
- **Biological graphs**: Rosalind tests overlap graphs (assembly), phylogenetic trees, gene networks

Here they're combined. Same algorithm, two contexts.

**Tags**: `[HR]` = HackerRank | `[R:ID]` = Rosalind problem ID

## Learning Objectives
- [ ] Implement BFS and DFS for both connected components and shortest paths
- [ ] Build overlap graphs from DNA reads
- [ ] Find Eulerian paths (genome assembly)
- [ ] Construct UPGMA and neighbor-joining phylogenetic trees
- [ ] Parse and manipulate Newick tree format

## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 00/01 (Python fundamentals), 01/06 (genome assembly context) |
| 📍 You Are Here | Module 08/03 — Graphs & Genome Assembly |
| ➡️ Next Steps | 08/04 (statistics & ML practical) |
| 🏁 Goal | Implement BFS/DFS, build a de Bruijn graph assembler, detect Eulerian paths, and solve HackerRank graph problems |

### 🆕 From Scratch? Start Here:
1. [NetworkX tutorial](https://networkx.org/documentation/stable/tutorial.html) — Python graph library
2. [Rosalind GRPH problem](https://rosalind.info/problems/grph/) — overlap graph construction
3. 00/01 (this repo) — Python fundamentals
4. This notebook — graphs & assembly
**Time:** 10–15h | **Difficulty:** ⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 00/01 (Python), 01/06 (genome assembly biology), 08/01 (k-mer strings that become graph nodes)
- Used by: 06/01 (GNNs generalize graph algorithms), 08/05 (phylogenetic trees are graphs)
- Parallel: 06/01 (graph neural networks — learned vs algorithmic graph methods)

In [ ]:
# Graphs Section 1 — BFS + DFS
from collections import deque, defaultdict

# --- BFS: shortest path ---
def bfs(graph, start, end):
    queue = deque([(start, [start])])
    visited = {start}
    while queue:
        node, path = queue.popleft()
        if node == end:
            return path
        for nbr in graph.get(node, []):
            if nbr not in visited:
                visited.add(nbr)
                queue.append((nbr, path+[nbr]))
    return None

# HackerRank: Shortest Reach (BFS)
graph = {1:[2,3], 2:[1,4], 3:[1,4], 4:[2,3,5], 5:[4]}
path = bfs(graph, 1, 5)
print(f"BFS path 1→5: {path}, distance={len(path)-1}")

# --- DFS: connected components (HackerRank Roads & Libraries) ---
def dfs_components(n, edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    visited = set()
    components = []
    def dfs(node, comp):
        visited.add(node)
        comp.append(node)
        for nbr in adj[node]:
            if nbr not in visited:
                dfs(nbr, comp)
    for node in range(1, n+1):
        if node not in visited:
            comp = []
            dfs(node, comp)
            components.append(comp)
    return components

comps = dfs_components(6, [(1,2),(2,3),(4,5)])
print(f"Components: {comps}")

In [ ]:
# Graphs Section 2 — Dijkstra + MST
import heapq

def dijkstra(graph, start):
    dist = {start: 0}
    heap = [(0, start)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist.get(u, float('inf')):
            continue
        for v, w in graph.get(u, []):
            nd = d + w
            if nd < dist.get(v, float('inf')):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist

# Weighted graph
wgraph = {
    1: [(2,6),(3,1)],
    2: [(1,6),(3,5),(4,5)],
    3: [(1,1),(2,5),(4,5),(5,4)],
    4: [(2,5),(3,5),(5,2)],
    5: [(3,4),(4,2)],
}
dist = dijkstra(wgraph, 1)
print("Dijkstra from node 1:", dist)

# Prim's MST
def prims_mst(graph, n):
    in_mst = {1}
    edges = [(w, 1, v) for v, w in graph.get(1, [])]
    heapq.heapify(edges)
    mst_weight = 0
    mst_edges = []
    while edges and len(in_mst) < n:
        w, u, v = heapq.heappop(edges)
        if v in in_mst:
            continue
        in_mst.add(v)
        mst_weight += w
        mst_edges.append((u, v, w))
        for nbr, nw in graph.get(v, []):
            if nbr not in in_mst:
                heapq.heappush(edges, (nw, v, nbr))
    return mst_weight, mst_edges

mst_w, mst_e = prims_mst(wgraph, 5)
print(f"MST weight: {mst_w}, edges: {mst_e}")

In [ ]:
# Graphs Section 3 — Overlap graph + Greedy assembly (Rosalind GRPH, LONG)
from collections import defaultdict

# --- Rosalind GRPH (overlap graph) ---
def overlap_graph(seqs, k=3):
    """Build directed graph: edge A→B if suffix of A of len k = prefix of B."""
    edges = []
    seq_list = list(seqs.items())
    for i, (id1, s1) in enumerate(seq_list):
        for j, (id2, s2) in enumerate(seq_list):
            if i != j and s1[-k:] == s2[:k]:
                edges.append((id1, id2))
    return edges

seqs = {'Rosalind_0498':'AAATAAA','Rosalind_2391':'AAATTTT',
        'Rosalind_2323':'TTTTCCC','Rosalind_0442':'AAATCCC','Rosalind_5013':'GGGTGGG'}
edges = overlap_graph(seqs, k=3)
print("Overlap graph edges (k=3):")
for e in edges:
    print(f"  {e[0]} → {e[1]}")

# --- Rosalind LONG (greedy shortest superstring) ---
def greedy_assembly(reads):
    while len(reads) > 1:
        best_overlap, best_i, best_j = 0, 0, 1
        for i in range(len(reads)):
            for j in range(len(reads)):
                if i == j: continue
                # Find overlap: longest suffix of reads[i] = prefix of reads[j]
                for k in range(min(len(reads[i]),len(reads[j])),0,-1):
                    if reads[i][-k:] == reads[j][:k]:
                        if k > best_overlap:
                            best_overlap, best_i, best_j = k, i, j
                        break
        merged = reads[best_i] + reads[best_j][best_overlap:]
        reads = [merged] + [reads[k] for k in range(len(reads)) if k!=best_i and k!=best_j]
    return reads[0]

reads = ["ATTAGACCTG","CCTGCCGGAA","AGACCTGCCG","GCCGGAATAC"]
assembly = greedy_assembly(reads)
print(f"Assembled: {assembly}")

In [ ]:
# Graphs Section 4 — De Bruijn + Euler path (Rosalind DBRU)
from collections import defaultdict

def de_bruijn(reads, k):
    graph = defaultdict(set)
    for read in reads:
        for i in range(len(read)-k+1):
            kmer = read[i:i+k]
            left, right = kmer[:-1], kmer[1:]
            graph[left].add(right)
    return {u: list(vs) for u, vs in graph.items()}

def eulerian_path(graph):
    """Hierholzer's algorithm for Eulerian path."""
    from collections import Counter
    in_deg = Counter(); out_deg = Counter()
    for u, vs in graph.items():
        out_deg[u] += len(vs)
        for v in vs:
            in_deg[v] += 1
    # Start at node with out > in
    start = next((n for n in out_deg if out_deg[n]-in_deg.get(n,0)==1), list(out_deg.keys())[0])
    g = {u: list(vs) for u, vs in graph.items()}
    stack = [start]; path = []
    while stack:
        v = stack[-1]
        if g.get(v):
            stack.append(g[v].pop())
        else:
            path.append(stack.pop())
    return path[::-1]

reads = ["CTTA","ACCA","TACC","GGCT","GCTT","TTAC"]
k = 3
g = de_bruijn(reads, k)
print("De Bruijn graph (k=3):")
for u, vs in sorted(g.items()):
    print(f"  {u} → {sorted(vs)}")
path = eulerian_path(g)
genome = path[0] + ''.join(n[-1] for n in path[1:])
print(f"Assembled: {genome}")

In [ ]:
# Graphs Section 5 — KMP + UPGMA + Resources
# Resources line for Graphs module
print("KEY RESOURCES — Module 08/03: Graphs + Assembly")
print()
refs = [
    "Rosalind GRPH, LONG, DBRU, TREE, NWCK, UPGMA",
    "HackerRank: Shortest Reach in a Graph, Roads and Libraries",
    "Cormen et al. CLRS: BFS (§22.2), DFS (§22.3), Dijkstra (§24.3)",
    "Pevzner & Compeau — Bioinformatics Algorithms (De Bruijn chapter)",
    "NetworkX library: networkx.org",
]
for r in refs:
    print(f"  {r}")

print()
print("COMPLEXITY CHEAT SHEET:")
algos = {
    "BFS/DFS": "O(V+E)",
    "Dijkstra (heap)": "O((V+E) log V)",
    "Prim's MST": "O(E log V)",
    "De Bruijn build": "O(n*L) where L=read length",
    "Eulerian path": "O(E)",
    "Greedy assembly": "O(n^2 * L)",
}
for alg, complexity in algos.items():
    print(f"  {alg}: {complexity}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Graph representations**: adjacency list O(V+E) space vs adjacency matrix O(V²) — when to use each
- **BFS/DFS complexity**: O(V+E) for adjacency list — know the proof; BFS gives shortest path in unweighted graphs
- **De Bruijn graph assembly**: nodes = (k-1)-mers, edges = k-mers — Eulerian path gives assembled sequence
- **Eulerian vs Hamiltonian**: Eulerian path (each edge once) is O(V+E) via Hierholzer; Hamiltonian path is NP-complete
- **MST algorithms**: Prim's O(E log V) with priority queue, Kruskal's O(E log E) — choose based on graph density

### 2️⃣ Must-Have Popular Resources
- 📘 CLRS ch. 22-25 — BFS, DFS, topological sort, shortest paths (Dijkstra, Bellman-Ford, Floyd-Warshall)
- 📘 Algorithm Design Manual ch. 5-7 (Skiena) — practical graph algorithms with real applications
- 📘 Bioinformatics Algorithms (Compeau & Pevzner) ch. 3 — de Bruijn graphs for genome assembly; free online
- 🎓 LeetCode graph study plan — 30 problems covering BFS/DFS/union-find/topological sort
- ⭐ GitHub [networkx/networkx](https://github.com/networkx/networkx) 14k★ — Python graph library for rapid prototyping

### 3️⃣ Practicals — Hands-On
- 💻 HackerRank: BFS Shortest Reach, Roads and Libraries (DFS/union-find), Dijkstra shortest path
- 💻 Rosalind: GRPH (overlap graph), LONG (shortest superstring), DBRU (de Bruijn from reads), UPGMA tree
- 💻 Implement Eulerian path detection — check in/out-degree, find path via Hierholzer's algorithm
- 🤗 HuggingFace: Protein interaction network datasets — build a PPI graph, find connected components
- 📊 Kaggle: [Predicting Molecular Properties](https://www.kaggle.com/c/champs-scalar-coupling) — graph-based molecular ML

### 4️⃣ Real-World Problems
- 🏭 SPAdes / Hifiasm genome assemblers: de Bruijn graphs over billions of k-mers — core industrial tool
- 🏭 Protein interaction networks at Genentech/Roche: BFS/community detection for pathway analysis
- 🏭 STRING-db: network-based disease gene discovery using graph centrality metrics
- 📊 Dataset: [STRING PPI database](https://string-db.org/) — 67M proteins, 20B interactions; export as edge list
- 🔬 Application: Metagenome assembly: de Bruijn graphs over millions of reads from environmental samples

### 5️⃣ Interview Tips
- ❓ Must know: Why de Bruijn (not overlap graph) for short reads — overlap graph has O(N²) edges; de Bruijn is O(N·k)
- ❓ Must know: Eulerian path existence conditions — exactly 0 or 2 vertices with odd degree (undirected), or in-degree ≠ out-degree for ≤2 nodes (directed)
- ❓ Must know: Dijkstra vs BFS — BFS for unweighted, Dijkstra for non-negative weights; Bellman-Ford for negative weights
- ⚠️ Common mistake: Using DFS for shortest path — DFS does NOT give shortest path in general graphs
- 💡 Pro tip: For assembly problems, draw the de Bruijn graph manually on paper first — the Eulerian path pattern becomes obvious

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [hifiasm](https://github.com/chhylp123/hifiasm) — state-of-the-art HiFi long-read assembler using de Bruijn + string graphs
- 🔥 Trending: [verkko](https://github.com/marbl/verkko) — telomere-to-telomere assembly; T2T human genome used this
- 🔥 Trending: [string-db.org](https://string-db.org/) — protein–protein interaction network ML for drug target discovery
- 🚀 Build this: A de Bruijn assembler that takes 100 simulated 10-mer reads from a known sequence and reconstructs it via Eulerian path

In [ ]:
# Resources for 08/03
print("Additional graph resources:")
print("  * Competitive Programming 3 — Steven Halim (graph chapters)")
print("  * Genome assembly survey: Nagarajan & Pop 2013 Nat Rev Genetics")
print("  * De novo assembly tools: SPAdes, ABySS, Velvet (use De Bruijn graphs)")

## Interview Cheat Sheet — Graphs & Trees

| Problem | Algorithm | Time | Space |
|---------|-----------|------|-------|
| Shortest path (unweighted) | BFS | O(V+E) | O(V) |
| Shortest path (weighted) | Dijkstra | O((V+E)logV) | O(V) |
| All-pairs shortest path | Floyd-Warshall | O(V³) | O(V²) |
| MST | Prim's / Kruskal's | O(E log V) | O(V) |
| Connected components | Union-Find or DFS | O(V+E) | O(V) |
| Topological sort | DFS or Kahn's | O(V+E) | O(V) |
| Eulerian path | Hierholzer's | O(V+E) | O(V+E) |
| Genome assembly | De Bruijn + Euler | O(n × L) | O(n × L) |
| Phylogenetic tree | UPGMA | O(n³) | O(n²) |

**Biological graph problems in interviews**:
- Overlap graph → find Hamiltonian path (NP-hard) vs Eulerian path (O(V+E)) → use De Bruijn
- Protein interaction network → connected components, centrality
- Phylogeny → tree edit distance, LCA queries

## Real World Problems

### Problem 1: Real Genome Assembly
Download E. coli reads from SRA (NCBI) and attempt assembly using SPAdes or implement a simplified De Bruijn assembler. Compare your assembled contigs to the reference.

**Resources**: [SPAdes assembler (GitHub)](https://github.com/ablab/spades) | [NCBI SRA datasets](https://www.ncbi.nlm.nih.gov/sra) | [Assembly benchmark (Kaggle)](https://www.kaggle.com/datasets/thomasnelson/genome-assembly)

### Problem 2: Protein Interaction Network Analysis
Download the STRING protein interaction network for yeast. Find: (1) degree distribution, (2) essential proteins (hubs with degree > 50), (3) shortest path between two metabolic pathway proteins.

**Resources**: [STRING network (HuggingFace)](https://huggingface.co/datasets/STRING-DB/string_db) | [NetworkX graph library (GitHub)](https://github.com/networkx/networkx)

### Problem 3: Rosalind Graph Track
GRPH → LONG → DBRU → CORR → KMER → KMP → EDIT → EDTA → GLOB → TRAN → PRSM → FULL

**Resources**: [rosalind.info graphs problems](https://rosalind.info/problems/list-view/) | [HackerRank graph theory](https://www.hackerrank.com/domains/algorithms?filters%5Bsubdomains%5D%5B%5D=graph-theory)

## Mastery Check

Before moving on, make sure you can:
1. solve representative problems without looking up the answer
2. state the time complexity correctly
3. explain why your solution is correct
4. identify which earlier notebook you should revisit if a problem still feels opaque


---
## 🐛 Debug Exercise — Graph Algorithm Bugs

Find and fix **3 bugs** in this BFS/DFS and topological sort.

<details><summary>Show bugs</summary>

**Bug 1:** BFS uses a list as a queue with `queue.pop(0)` — this is O(n) per dequeue. Should use `collections.deque` with `popleft()`. (Correctness bug: for very large graphs this makes BFS O(n²) and effectively wrong for timed problems.)

**Bug 2:** DFS for cycle detection: the `visited` set marks nodes globally, but cycle detection needs to distinguish "currently in recursion stack" from "already finished". Need a separate `in_stack` set.

**Bug 3:** Topological sort uses DFS but appends node to result BEFORE visiting neighbors (preorder), should append AFTER (postorder), then reverse.

</details>

In [ ]:
from collections import defaultdict

def bfs_buggy(graph, start):
    visited = set()
    queue = [start]  # BUG 1: list instead of deque; pop(0) is O(n)
    order = []
    while queue:
        node = queue.pop(0)   # should be popleft() on deque
        if node not in visited:
            visited.add(node)
            order.append(node)
            queue.extend(graph.get(node, []))
    return order

def has_cycle_buggy(graph):
    visited = set()
    # BUG 2: no in_stack tracking — can't detect cycles correctly
    def dfs(node):
        if node in visited: return True  # wrong: finished nodes look like cycles
        visited.add(node)
        for neighbor in graph.get(node, []):
            if dfs(neighbor): return True
        return False
    return any(dfs(n) for n in graph if n not in visited)

def topo_sort_buggy(graph):
    visited = set()
    result = []
    def dfs(node):
        visited.add(node)
        result.append(node)          # BUG 3: preorder append — should be postorder
        for n in graph.get(node, []):
            if n not in visited: dfs(n)
    for n in graph:
        if n not in visited: dfs(n)
    return result                    # and should return list(reversed(result))

g = {1:[2,3], 2:[4], 3:[4], 4:[]}
print(f"BFS from 1: {bfs_buggy(g, 1)}")
print(f"Expected:   [1, 2, 3, 4] (BFS level order)")

dag = {1:[2], 2:[3], 3:[], 4:[2]}
cyc = {1:[2], 2:[3], 3:[1]}  # cycle
print(f"\nCycle detection (no cycle): {has_cycle_buggy(dag)} (expected: False)")
print(f"Cycle detection (has cycle): {has_cycle_buggy(cyc)} (expected: True)")

print(f"\nTopo sort: {topo_sort_buggy(dag)}")
print(f"Expected:  [1, 4, 2, 3] or similar valid ordering (4 before 2 possible)")
